In [ ]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
import cvxpy as cp

In [ ]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = self.download(tickers, start, end, interval)
      bench_data = self.download([benchmark], start, end, interval)

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()
    self.universe = returns_raw.columns

    return returns_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [ ]:
class Optimizer:
  def __init__(
      self,
      es_prct,
      turnover_penalty,
      risk_penalty,
      tail_penalty,
      debug:bool=False,
      **kwargs
  ):
    self.debug = debug
    self.es_prct = es_prct
    self.turnover_penalty = turnover_penalty
    self.risk_pen = risk_penalty
    self.

  def optimize_lambda(self, A, b, k_eq, k_ineq):
    K = k_eq + k_ineq
    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exp_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exp_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    return lambda_var.value

  def optimize(self, S, N, returns, mu, cov, w_prev=None):
    R = returns.values
    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    es = zeta + (1 / ((1-self.es_prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0
    ]

    ex_r = mu @ w

    if turnover_penalty is not None and w_prev is not None:
      turnover_penalty = self.turnover_penalty * cp.sum((w - w_prev)**2)

    else:
      turnover_penalty = 0

    risk_term = cp.quad_form(w, cov)
    obj_fn = cp.Maximize(ex_r - risk_term - es - turnover_penalty)

    prob = cp.Problem(obj_fn, constraints)
    prob.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"GMV optimization failed: {problem.status}")

    return w.value







In [ ]:
class Portfolio(Optimizer, DataStore):
  def __init__(self, debug:bool=False, **kwargs):
    self.debug = debug
    super().__init__(debug, **kwargs)

  def _add(self, A, b, vtype):
    if vtype == "ineq":
      self.A_ineq.append(A)
      self.b_ineq.append(b)
    elif vtype == "eq":
      self.A_eq.append(A)
      self.b_eq.append(b)

  def get_views(self, views, returns, p):
    self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = [], [], [], []

    for view in views:
      if view["view"] == "mean":
        idx = returns.columns.get_loc(view["ticker"])

        self._add(returns[:, idx], view["value"], view["type"])

      elif view["view"] == "volatility":
        idx = returns.columns.get_loc(view["ticker"])
        R = returns[:, idx]
        mu = np.sum(p*R)
        var = (R - mu)**2

        self._add(var, view["value"]**2, view["type"])

      elif view["view"] == "relative":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        A_rel = R[:, idx1] - R[:, idx2]

        self._add(A_rel, view["value"], view["type"])

      elif view["view"] == "correlation":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        mu_1 = np.sum(p*R[:, idx1])
        mu_2 = np.sum(p*R[:, idx2])

        vol_1 = np.sqrt(np.sum(p*(R[:, idx1] - mu_1)**2))
        vol_2 = np.sqrt(np.sum(p*(R[:, idx2] - mu_2)**2))
        rho = view["value"]
        cov_rarget = rho*vol_1*vol_2

        A_corr = (R[:, idx1] - mu_1)*(R[:, idx2] - mu_2)
        self._add(A_corr, cov_rarget, view["type"])

    def generate_portfolio(self, returns, views, p=None):
      S, N = returns.shape
      p = p if p is not None else np.ones(S)/S
      self.get_views(views, returns, p)

    def solve_entropy_pooling(self, R, p=None, A_eq=None, b_eq=None, A_ineq=None, b_ineq=None):
      if p is None:
        p = np.ones(S) / S

      A_list, b_list, constraints = [], [], []
      k_eq = 0

      if A_eq is not np.None and b_eq is not np.None:
        A_eq_clean = np.atleast_2d(A_eq)
        b_eq_clean = np.atleast_1d(b_eq)
        A_list.append(A_eq_clean)
        b_list.append(b_eq_clean)
        k_eq = A_eq_clean.shape[0]

      k_ineq = 0
      if A_ineq is not np.None and b_ineq is not np.None:
        A_ineq_clean = np.atleast_2d(A_ineq)
        b_ineq_clean = np.atleast_1d(b_ineq)
        A_list.append(A_ineq_clean)
        b_list.append(b_ineq_clean)
        k_ineq = A_ineq_clean.shape[0]

      A = np.vstack(A_list)
      b = np.concatenate(b_list)

      opt_lambda = self.optimize_lambda(A, b, k_eq, k_ineq)

      q = p * np.exp(-opt_lambda @ A)
      q /= np.sum(q)
      mu_post = q @ R
      R_dev = R - mu_post
      cov_post = (R_dev.T * q) @ R_dev

      return q, mu_post, cov_post

  def optimize_portfolio(self, returns, views, p=None):
    S, N = returns.shape

    p = p if p is not None else np.ones(S)/S
    self.get_views(views, returns, p)
    w_opt = optimize_w(S, N, returns, mu, cov)






In [ ]:
ptf = Portfolio(debug=False)

In [ ]:
data, benchmark = ptf.get_data(
    tickers=tickers,
    start="2020-01-01",
    end="2026-01-01",
    interval="1d"
)